# Soft clusters: Gaussian mixtures
**Machine Learning · Week 6 (Learning without labels) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/ml/w6d2.html

A **Gaussian mixture** describes data as a few bell-shaped blobs, each with its own center, shape and
weight. **EM** fits it by alternating two steps: give every point a probability for every blob (E),
then re-fit every blob from its weighted points (M).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
from matplotlib.patches import Ellipse

rng = np.random.default_rng(0)
# three long, thin stripes: the data k-means got wrong yesterday
direction, normal = np.array([np.cos(0.5), np.sin(0.5)]), np.array([-np.sin(0.5), np.cos(0.5)])
t = rng.normal(0, 1.0, 300)
u = rng.normal(0, 0.08, 300) + np.repeat([-0.8, 0.0, 0.8], 100)
X = t[:, None] * direction + u[:, None] * normal

def em(X, k, steps=30, seed=5):
    rng = np.random.default_rng(seed)
    means = X[rng.choice(len(X), k, replace=False)]
    covs = np.array([np.eye(2) * 0.35] * k)
    weights = np.full(k, 1 / k)
    history = []
    for _ in range(steps):
        # E-step: responsibilities r[i, j] = P(blob j | point i), by Bayes' rule
        dens = np.column_stack([w * multivariate_normal(m, c).pdf(X) for w, m, c in zip(weights, means, covs)])
        history.append(np.log(dens.sum(axis=1)).mean())          # average log-likelihood
        r = dens / dens.sum(axis=1, keepdims=True)
        # M-step: weighted counts, means and covariances
        n = r.sum(axis=0)
        weights = n / len(X)
        means = (r.T @ X) / n[:, None]
        covs = np.array([((r[:, j, None] * (X - means[j])).T @ (X - means[j])) / n[j] + 1e-6 * np.eye(2)
                         for j in range(k)])
    return weights, means, covs, r, history

weights, means, covs, r, history = em(X, 3)
palette = np.array([[0.48, 0.36, 0.98], [0.09, 0.66, 0.55], [1.0, 0.54, 0.31]])

def draw_ellipses(ax, means, covs):
    for j, (m, c) in enumerate(zip(means, covs)):
        vals, vecs = np.linalg.eigh(c)
        angle = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
        for s, ls in [(2, "-"), (4, "--")]:                      # 1 and 2 standard deviations (width = 2 sd)
            ax.add_patch(Ellipse(m, s * np.sqrt(vals[1]), s * np.sqrt(vals[0]), angle=angle,
                                 fill=False, ec=palette[j], lw=2, ls=ls))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.5, 3.6), layout="constrained")
ax1.scatter(X[:, 0], X[:, 1], c=r @ palette, s=12)
draw_ellipses(ax1, means, covs)
ax1.set_aspect("equal")
ax1.set_title("EM from scratch: three stretched blobs")
ax2.plot(history, "o-", ms=4, color="#17a98c")
ax2.set_xlabel("EM step")
ax2.set_ylabel("average log-likelihood")
ax2.set_title("It never goes down")
plt.show()
print("weights:", weights.round(3))

## scikit-learn, and the comparison with k-means

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans

# start from random data points: the default start (k-means' own answer) can inherit k-means' mistake here
gmm = GaussianMixture(n_components=3, covariance_type="full", init_params="random_from_data",
                      n_init=10, random_state=0).fit(X)
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), layout="constrained")
for ax, labels, title in [(axes[0], km.labels_, "k-means: round clusters"), (axes[1], gmm.predict(X), "Gaussian mixture: stretched blobs")]:
    ax.scatter(X[:, 0], X[:, 1], c=palette[labels], s=12)
    ax.set_aspect("equal")
    ax.set_title(title)
draw_ellipses(axes[1], gmm.means_, gmm.covariances_)
plt.show()

## Soft memberships
Three overlapping blobs. `predict_proba` gives each point a probability for each blob; points
between blobs are honestly unsure.

In [ ]:
from sklearn.datasets import make_blobs

X2, _ = make_blobs(n_samples=300, centers=[(0, 0), (2.6, 0.6), (1.1, 2.6)], cluster_std=0.85, random_state=1)
gmm2 = GaussianMixture(n_components=3, random_state=0).fit(X2)
membership = gmm2.predict_proba(X2)                      # one row per point, one column per blob
unsure = membership.max(axis=1) < 0.8

plt.scatter(X2[:, 0], X2[:, 1], c=membership @ palette, s=18)
plt.scatter(X2[unsure, 0], X2[unsure, 1], s=90, facecolors="none", edgecolors="#2c2842", linewidths=1)
plt.title(f"Soft clusters: {unsure.sum()} circled points have no blob above 80%")
plt.show()
print("a confident point:", membership[np.argmax(membership.max(axis=1))].round(3))
print("an unsure point  :", membership[np.argmin(membership.max(axis=1))].round(3))

## How many blobs? BIC
More blobs always fit the training data better. The **Bayesian information criterion** adds a
penalty for every parameter: $\text{BIC} = -2\log L + (\text{number of parameters})\log n$. Lower is better.
Back to the stripes:

In [ ]:
ks = range(1, 8)
bics = [GaussianMixture(n_components=k, init_params="random_from_data", n_init=10, random_state=0).fit(X).bic(X)
        for k in ks]
plt.plot(ks, bics, "o-", color="#7b5cfa")
plt.xlabel("number of blobs k")
plt.ylabel("BIC (lower is better)")
plt.title(f"Lowest BIC at k = {int(np.argmin(bics)) + 1}")
plt.show()

## Your turn: the E-step by hand
Write `responsibilities(X, weights, means, covs)`: for every point and every blob,
$r_{ij} = \frac{\pi_j\,\mathcal{N}(x_i; \mu_j, \Sigma_j)}{\sum_l \pi_l\,\mathcal{N}(x_i; \mu_l, \Sigma_l)}$.
Check it against scikit-learn: with the fitted `gmm2`, your answer should match `gmm2.predict_proba(X2)`.

In [ ]:
def responsibilities(X, weights, means, covs):
    # TODO: one column per blob: weight * multivariate_normal(mean, cov).pdf(X)
    # TODO: divide every row by its sum
    return np.full((len(X), len(weights)), 1 / len(weights))

mine = responsibilities(X2, gmm2.weights_, gmm2.means_, gmm2.covariances_)
print("rows sum to 1:", np.allclose(mine.sum(axis=1), 1))
print("matches scikit-learn:", np.allclose(mine, gmm2.predict_proba(X2), atol=1e-6))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def responsibilities(X, weights, means, covs):
    dens = np.column_stack([w * multivariate_normal(m, c).pdf(X) for w, m, c in zip(weights, means, covs)])
    return dens / dens.sum(axis=1, keepdims=True)

mine = responsibilities(X2, gmm2.weights_, gmm2.means_, gmm2.covariances_)
print("rows sum to 1:", np.allclose(mine.sum(axis=1), 1))
print("matches scikit-learn:", np.allclose(mine, gmm2.predict_proba(X2), atol=1e-6))
print("That's Bayes' rule: prior (the weight) times likelihood (the bell curve), normalized.")